In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import nltk

plt.style.use('ggplot')


In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "Reviews.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "snap/amazon-fine-food-reviews",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)


In [ ]:
df.head()

In [ ]:
df['Text'][0]

In [ ]:
df.shape

In [ ]:
df = df.head(500)
df.shape

In [ ]:
df['Score'].value_counts().sort_index().plot(kind='bar', title="Count of review by stars", figsize=(10,5))

In [ ]:
example = df['Text'][50]
print(example)

In [ ]:
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
tokens = nltk.word_tokenize(example)
tokens

In [ ]:
tagged = nltk.pos_tag(tokens)
tagged

In [ ]:
nltk.download("maxent_ne_chunker_tab")
nltk.download("words")

In [ ]:
entities = nltk.chunk.ne_chunk(tagged)
entities.pprint()

## Step 1 : VADER Sentiment Scoring

we will use the VADER (Valence Aware Dictionary and sEntiment Reasoner) sentiment analysis tool from the NLTK library to analyze the sentiment of the reviews in our dataset. VADER is specifically designed for sentiment analysis of social media text and is effective in handling emojis, slang, and other informal language.

This uses "Bag of Words" approach:
1. Stop words are removed.
2. Each word scored and combined to a total score.

In [ ]:
nltk.download('vader_lexicon')

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
from tqdm import tqdm

In [ ]:
sia = SentimentIntensityAnalyzer()

In [ ]:
sia

In [ ]:
sia.polarity_scores("I am so happy!")

Here neg is a negative word, pos is a positive word, neu is a neutral word and compound is the overall score. Compound score is calculated by summing the valence scores of each word in the lexicon, adjusted according to the rules, and then normalized to be between -1 (most extreme negative) and +1 (most extreme positive).

In [ ]:
sia.polarity_scores("This is the worst thing ever")

In [ ]:
sia.polarity_scores(example)

In [ ]:
# Run the polarity score on entire dataset
df

In [ ]:
res = {}
for i, row in tqdm(df.iterrows(), total=df.shape[0]):
    text = row['Text']
    myid = row['Id']
    res[myid] = sia.polarity_scores(text)


In [ ]:
res

In [ ]:
vaders = pd.DataFrame(res).T
vaders.head()

In [ ]:
vaders = vaders.reset_index().rename(columns={'index':'Id'})
vaders = vaders.merge(df, how='left')
vaders

In [ ]:
sns.barplot(data=vaders, x='Score', y='compound')
plt.title("Compound score by Amazon review stars")
plt.show()

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(20,5))
sns.barplot(data=vaders, x='Score', y='pos', ax=axes[0])
sns.barplot(data=vaders, x='Score', y='neg', ax=axes[1])
sns.barplot(data=vaders, x='Score', y='neu', ax=axes[2])
axes[0].set_title("Positive score by Amazon review stars")
axes[1].set_title("Negative score by Amazon review stars")
axes[2].set_title("Neutral score by Amazon review stars")
plt.show()

## Step 2. Roberta Pretrained Model
1. Use a model trained of a large corpus of data.
2. Transformer model accounts for the words but also the context related to other words.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax

In [ ]:
MODEL = f"cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

In [ ]:
example

In [ ]:
sia.polarity_scores(example)

In [ ]:
# Run on Roberta Model

encoded_texts = tokenizer(example, return_tensors='pt')
encoded_texts

In [ ]:
output = model(**encoded_texts)
output

In [ ]:
scores = output[0][0].detach().numpy()
scores

In [ ]:
scores = softmax(scores)
scores

In [ ]:
scores_dict = {
    'roberta_neg': scores[0],
    'roberta_neu': scores[1],
    'roberta_pos': scores[2]
}
scores_dict

## Step 3 : Combine and Compare

In [ ]:
def polar_scores_reberta(example):
    # Truncate text to fit within the model's maximum sequence length (typically 512)
    encoded_texts = tokenizer(example, return_tensors='pt')
    output = model(**encoded_texts)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    scores_dict = {
        'roberta_neg': scores[0],
        'roberta_neu': scores[1],
        'roberta_pos': scores[2]
    }
    return scores_dict

In [ ]:
res = {}
for i, row in tqdm(df.iterrows(), total=df.shape[0]):
    try:
        text = row['Text']
        myid = row['Id']
        vader_result = sia.polarity_scores(text)
        vader_result_rename = {}
        for k, v in vader_result.items():
            vader_result_rename[f"vader_{k}"] = v
        roberta_result = polar_scores_reberta(text)
        both = {**vader_result_rename, **roberta_result}
        res[myid] = both
    except RuntimeError:
        print(f'Broke for id {myid}')

In [ ]:
len(df['Text'].iloc[186]), len(df['Text'].iloc[82])

In [ ]:
results_df = pd.DataFrame(res).T
results_df = results_df.reset_index().rename(columns={'index':'Id'})
results_df = results_df.merge(df, how='left')

In [ ]:
results_df.head()

In [ ]:
sns.pairplot(data=results_df, vars=['vader_neg', 'vader_neu', 'vader_pos', 'roberta_neg', 'roberta_neu', 'roberta_pos'], hue='Score', palette='tab10')
plt.show()

## Step 4 : Review Example

* positive 1-Star and Negative 5-Star Reviews

Lets look at some examples where the model scoring and review score differ the most

In [ ]:
results_df.query("Score == 1").sort_values("roberta_pos", ascending=False)["Text"].values[0]

In [ ]:
results_df.query("Score == 1").sort_values("vader_pos", ascending=False)[["Text","vader_pos"]].values[0]

In [ ]:
results_df.query("Score == 5").sort_values("roberta_neg", ascending=False)["Text"].values[0]

In [ ]:
results_df.query("Score == 5").sort_values("vader_neg", ascending=False)["Text"].values[0]

In [ ]:
from transformers import pipeline

In [ ]:
senti_pipeline = pipeline('sentiment-analysis')

In [ ]:
senti_pipeline("I am so happy!")

In [ ]:
senti_pipeline("This is the worst thing ever")

In [ ]:
senti_pipeline("this was sooooo deliscious but too bad i ate em too fast and gained 2 pds! my fault")

In [ ]:
senti_pipeline("So we cancelled the order.  It was cancelled without any problem.  That is a positive note...")

In [ ]:
senti_pipeline("I felt energized within five minutes, but it lasted for about 45 minutes. I paid $3.99 for this drink. I could have just drunk a cup of coffee and saved my money.")

In [ ]:
results_df.query("Score == 1 and roberta_pos > 0.5").sort_values("roberta_pos", ascending=False)[["Text", "roberta_pos"]]

In [ ]:
results_df.query("Score == 1 and vader_pos > 0.5").sort_values("vader_pos", ascending=False)[["Text", "vader_pos"]]

In [ ]:
results_df.query("Score == 5 and roberta_neg > 0.5").sort_values("roberta_neg", ascending=False)[["Text", "roberta_neg"]]

In [ ]:
results_df.query("Score == 5 and vader_neg > 0.5").sort_values("vader_neg", ascending=False)[["Text", "vader_neg"]]

In [ ]:
results_df.iloc[69]